# 🚀 Notebook 6: Full Pipeline — PDF to Ontology

End-to-end demo: **PDF files → Text Extraction → NLP/LLM Extraction → OWL Ontology → Validation → SPARQL → Visualization**

This notebook runs the complete pipeline on your own PDF documents.

---

## 0. Setup

In [ ]:
!pip install -q PyMuPDF pdfplumber pytesseract Pillow spacy scikit-learn owlready2 rdflib networkx matplotlib anthropic tqdm
!apt-get install -q tesseract-ocr > /dev/null 2>&1
!python -m spacy download en_core_web_sm -q

!git clone https://github.com/YOUR_USERNAME/ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'ontology-builder')

import os, json, logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
os.makedirs('output', exist_ok=True)
print('Setup complete!')

## 1. Upload PDFs

In [ ]:
# Option A: Upload files
from google.colab import files
os.makedirs('pdfs', exist_ok=True)

print('Upload your PDF files:')
uploaded = files.upload()

for fn, content in uploaded.items():
    with open(f'pdfs/{fn}', 'wb') as f:
        f.write(content)
    print(f'  Saved: pdfs/{fn} ({len(content)} bytes)')

# Option B: Mount Google Drive (uncomment below)
# from google.colab import drive
# drive.mount('/content/drive')
# PDF_DIR = '/content/drive/MyDrive/your-pdf-folder'

## 2. Configuration

In [ ]:
# ── Pipeline Configuration ──────────────────────────────

CONFIG = {
    # Input
    'pdf_directory': 'pdfs/',
    
    # Extraction method: 'nlp', 'llm', or 'hybrid'
    'method': 'nlp',
    
    # NLP settings
    'spacy_model': 'en_core_web_sm',
    'min_concept_frequency': 2,
    'max_concepts': 300,
    'min_relation_confidence': 0.3,
    
    # LLM settings (only used if method='llm' or 'hybrid')
    'anthropic_api_key': '',  # Set your key here or use env var
    'llm_model': 'claude-sonnet-4-20250514',
    'domain_context': '',  # e.g., 'biomedical research' or 'legal contracts'
    
    # Ontology output
    'ontology_iri': 'http://example.org/my-ontology',
    'ontology_name': 'MyOntology',
    'output_format': 'rdfxml',
    'output_path': 'output/ontology.owl',
    
    # Validation
    'run_reasoner': True,
    'reasoner': 'hermit',
}

print('Configuration:')
for k, v in CONFIG.items():
    if 'key' not in k.lower():
        print(f'  {k}: {v}')

## 3. Step 1 — Extract Text from PDFs

In [ ]:
from src.pdf_extractor import PDFExtractor

extractor = PDFExtractor(ocr_enabled=True)
documents = extractor.extract_from_directory(CONFIG['pdf_directory'])

print(f'\nExtracted {len(documents)} documents:')
total_chars = 0
for doc in documents:
    chars = len(doc.full_text)
    total_chars += chars
    print(f'  {doc.filename}: {doc.num_pages} pages, {chars:,} chars')
print(f'\nTotal: {total_chars:,} characters')

## 4. Step 2 — Preprocess Text

In [ ]:
from src.preprocessor import TextPreprocessor

preprocessor = TextPreprocessor()
segments = preprocessor.preprocess_documents(documents)

print(f'\nTotal segments: {len(segments)}')
headings = [s for s in segments if s.segment_type == 'heading']
print(f'Headings detected: {len(headings)}')
if headings:
    print('Sample headings:')
    for h in headings[:10]:
        print(f'  • {h.text[:60]}')

## 5. Step 3 — Extract Knowledge

In [ ]:
from src.concept_extractor import ConceptExtractor
from src.relation_extractor import RelationExtractor

concepts = []
relations = []
llm_knowledge = None

# ── NLP Extraction ──
if CONFIG['method'] in ('nlp', 'hybrid'):
    print('Running NLP extraction...')
    
    concept_ext = ConceptExtractor(
        spacy_model=CONFIG['spacy_model'],
        min_frequency=CONFIG['min_concept_frequency'],
        max_concepts=CONFIG['max_concepts'],
    )
    concepts = concept_ext.extract(segments)
    
    rel_ext = RelationExtractor(
        spacy_model=CONFIG['spacy_model'],
        min_confidence=CONFIG['min_relation_confidence'],
    )
    relations = rel_ext.extract(segments, concepts)
    
    print(f'  NLP: {len(concepts)} concepts, {len(relations)} relations')

# ── LLM Extraction ──
if CONFIG['method'] in ('llm', 'hybrid'):
    api_key = CONFIG['anthropic_api_key'] or os.environ.get('ANTHROPIC_API_KEY')
    if api_key:
        print('Running LLM extraction...')
        from src.llm_extractor import LLMExtractor
        
        llm_ext = LLMExtractor(
            api_key=api_key,
            model=CONFIG['llm_model'],
            domain_context=CONFIG['domain_context'],
        )
        llm_knowledge = llm_ext.extract_from_documents(documents)
        print(f'  LLM: {len(llm_knowledge.concepts)} concepts, '
              f'{len(llm_knowledge.relations)} relations, '
              f'{len(llm_knowledge.attributes)} attributes')
    else:
        print('  Skipping LLM extraction (no API key set)')

print('\nExtraction complete!')

### Inspect Extracted Knowledge

In [ ]:
import pandas as pd

if concepts:
    df_concepts = pd.DataFrame([
        {'Name': c.name, 'Label': c.label, 'Frequency': c.frequency, 'Parent': c.parent}
        for c in concepts
    ])
    print(f'\nTop 20 Concepts (NLP):')
    display(df_concepts.head(20))

if relations:
    df_rels = pd.DataFrame([
        {'Subject': r.subject, 'Predicate': r.predicate, 'Object': r.object,
         'Confidence': f'{r.confidence:.2f}', 'Type': r.relation_type}
        for r in relations
    ])
    print(f'\nTop 20 Relations (NLP):')
    display(df_rels.head(20))

## 6. Step 4 — Build the Ontology

In [ ]:
from src.ontology_builder import OntologyBuilder

builder = OntologyBuilder(
    iri=CONFIG['ontology_iri'],
    name=CONFIG['ontology_name'],
)

# Add NLP-extracted knowledge
if concepts:
    builder.add_concepts(concepts)
if relations:
    builder.add_relations(relations)

# Add LLM-extracted knowledge (if available)
if llm_knowledge:
    builder.from_llm_output(llm_knowledge)

# Save
builder.save(CONFIG['output_path'], format=CONFIG['output_format'])

stats = builder.get_stats()
print('\nOntology Built!')
print(f'  Classes:           {stats["classes"]}')
print(f'  Object Properties: {stats["object_properties"]}')
print(f'  Data Properties:   {stats["data_properties"]}')
print(f'  Individuals:       {stats["individuals"]}')
print(f'  Saved to:          {CONFIG["output_path"]}')

## 7. Step 5 — Validate

In [ ]:
if CONFIG['run_reasoner']:
    from src.validator import OntologyValidator
    
    validator = OntologyValidator(reasoner=CONFIG['reasoner'])
    report = validator.validate(builder.onto)
    
    print(report.summary())
else:
    print('Reasoner validation skipped.')

## 8. Step 6 — Query with SPARQL

In [ ]:
from src.query_engine import QueryEngine

engine = QueryEngine(CONFIG['output_path'])
print(f'Loaded {engine.count_triples()} triples\n')

# All classes
classes = engine.get_all_classes()
print(f'Classes ({len(classes)}):')
for c in classes[:15]:
    name = c['class'].split('/')[-1].split('#')[-1]
    print(f'  • {name}')

# Hierarchy
print(f'\nClass Hierarchy:')
hierarchy = engine.get_class_hierarchy()
for row in hierarchy[:20]:
    child = row['child'].split('/')[-1].split('#')[-1]
    parent = row['parent'].split('/')[-1].split('#')[-1]
    print(f'  {child} ⊑ {parent}')

## 9. Step 7 — Visualize

In [ ]:
from src.visualizer import OntologyVisualizer

viz = OntologyVisualizer(CONFIG['output_path'])
summary = viz.get_summary()
print(f'Graph: {summary["nodes"]} nodes, {summary["edges"]} edges')

viz.plot_hierarchy(
    figsize=(18, 14),
    title=f'{CONFIG["ontology_name"]} — Ontology Graph',
    save_path='output/ontology_graph.png'
)

# Export for WebVOWL
viz.export_for_webvowl('output/ontology.ttl')

## 10. Download Results

In [ ]:
from google.colab import files

output_files = [f for f in os.listdir('output') if os.path.isfile(f'output/{f}')]
print('Output files:')
for f in output_files:
    size = os.path.getsize(f'output/{f}')
    print(f'  {f} ({size:,} bytes)')

# Download the ontology
print('\nDownloading ontology file...')
files.download(CONFIG['output_path'])

# Uncomment to download all outputs:
# for f in output_files:
#     files.download(f'output/{f}')

---

## 🎉 Done!

You now have a complete OWL ontology built from your PDF documents.

**Next steps:**
- Open the `.owl` file in [Protégé](https://protege.stanford.edu/) for manual editing
- Upload the `.ttl` file to [WebVOWL](http://www.visualdataweb.de/webvowl/) for interactive visualization
- Deploy a SPARQL endpoint with [Apache Jena Fuseki](https://jena.apache.org/documentation/fuseki2/)
- Iterate: adjust thresholds, add domain context, re-run with LLM extraction